# Guide · Neural network from scratch: NumPy → PyTorch

> 🇪🇸 Versión en español: [`01_red_neuronal_desde_cero.ipynb`](01_red_neuronal_desde_cero.ipynb)

In this guide **you** build a **multi-layer perceptron (MLP)** with NumPy only: activations, loss,
forward pass, **hand-derived backpropagation** and gradient descent. Then you write the same network in
PyTorch and check that **its gradients match yours**.

### How to use this guide
- Cells marked ✍️ contain `# TODO` gaps. Replace each `raise NotImplementedError` with your code.
- Right after each one there is a ✅ cell that **checks** your implementation. If it passes, move on;
  if it fails, the message tells you what to look at.
- The theory in section 2 has every equation you need. Try not to look up the solution:
  getting the matrix shapes wrong is part of learning.

Goals:
1. Understand what each layer computes and why a **non-linearity** is needed.
2. Derive and implement the **chain rule** in matrix form.
3. Verify your implementation with a **numerical gradient check**.
4. See what PyTorch automates (`autograd`, `nn.Module`, `optim`) and what it does not.

In [ ]:
# Libraries
#=====================================================
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
print("numpy", np.__version__)

## 1. Data: three spirals

A classic problem that **no linear model** can solve: three classes wound into spirals.
This cell is already complete; run it and look at the data.

In [ ]:
def spirals(n_per_class=200, n_classes=3, noise=0.2, seed=0):
    """Return X with shape (n, 2) and labels y with shape (n,) in 0..n_classes-1."""
    g = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        r = np.linspace(0.0, 1.0, n_per_class)                       # radius
        t = np.linspace(k * 4, (k + 1) * 4, n_per_class) + g.normal(0, noise, n_per_class)  # angle
        X.append(np.column_stack([r * np.sin(t), r * np.cos(t)]))
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)

X, y = spirals()

# Train / test split (80 / 20)
idx = rng.permutation(len(X))
cut = int(0.8 * len(X))
X_train, y_train = X[idx[:cut]], y[idx[:cut]]
X_test, y_test = X[idx[cut:]], y[idx[cut:]]
print(X_train.shape, X_test.shape, np.bincount(y_train))

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=12)
plt.title("Three spirals"); plt.axis("equal"); plt.show()

## 2. The math

For a batch $X \in \mathbb{R}^{N \times D}$ and a network with layers $\ell = 1, \dots, L$:

$$
Z^{(\ell)} = A^{(\ell-1)} W^{(\ell)} + b^{(\ell)}, \qquad
A^{(\ell)} = \mathrm{ReLU}\big(Z^{(\ell)}\big), \qquad A^{(0)} = X
$$

The last layer has no ReLU: it outputs **logits** $Z^{(L)} \in \mathbb{R}^{N \times K}$, which we turn into
probabilities with **softmax** and compare with the true class through the **cross-entropy**:

$$
P_{ik} = \frac{e^{Z_{ik}}}{\sum_j e^{Z_{ij}}}, \qquad
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P_{i, y_i}
$$

### Backpropagation (the chain rule in matrix form)

Let $Y$ be the one-hot label matrix. Softmax combined with cross-entropy has a surprisingly simple gradient:

$$
\frac{\partial \mathcal{L}}{\partial Z^{(L)}} = \frac{1}{N}\,(P - Y)
$$

From there, for each layer (last to first), with $\delta^{(\ell)} = \partial \mathcal{L} / \partial Z^{(\ell)}$:

$$
\frac{\partial \mathcal{L}}{\partial W^{(\ell)}} = A^{(\ell-1)\top} \delta^{(\ell)}, \qquad
\frac{\partial \mathcal{L}}{\partial b^{(\ell)}} = \sum_{i} \delta^{(\ell)}_{i,:}, \qquad
\delta^{(\ell-1)} = \big(\delta^{(\ell)} W^{(\ell)\top}\big) \odot \mathbb{1}\big[Z^{(\ell-1)} > 0\big]
$$

$\odot$ is the element-wise product and $\mathbb{1}[Z>0]$ is the derivative of ReLU.
Note that **the shapes line up**: $A^{(\ell-1)\top}$ is $(n_{\ell-1} \times N)$ and $\delta^{(\ell)}$ is
$(N \times n_\ell)$, so the product has the shape of $W^{(\ell)}$. Checking shapes is the best way to
debug backprop.

> 💡 Before coding, try deriving $\partial \mathcal{L} / \partial Z^{(L)} = (P - Y)/N$ on paper.

## 3. The building blocks in NumPy

### 3.1 ✍️ Activation, softmax and loss
- `relu`: element-wise $\max(0, z)$.
- `softmax`: subtract each row's maximum before `exp` (**stable softmax**). The result is identical,
  but you avoid overflow with large logits. Hint: `keepdims=True`.
- `cross_entropy`: return the mean loss **and** its gradient with respect to the logits, $(P - Y)/N$.
  Hint: `P[np.arange(n), y]` picks the probability of the correct class in each row.

In [ ]:
def relu(z):
    # TODO: element-wise max(0, z)
    raise NotImplementedError

def softmax(z):
    """Row-wise softmax. z has shape (N, K)."""
    # TODO: subtract each row's maximum, apply exp and normalize so every row sums to 1
    raise NotImplementedError

def cross_entropy(logits, y):
    """Mean loss and its gradient with respect to the logits: (P - Y) / N."""
    n = len(y)
    P = softmax(logits)
    # TODO 1: loss = mean of -log(probability of the correct class)  (add 1e-12 inside the log)
    # TODO 2: dlogits = (P - Y) / n without building Y: copy P and subtract 1 at the correct column of each row
    raise NotImplementedError

In [ ]:
# ✅ Check 3.1
assert np.array_equal(relu(np.array([-2.0, 0.0, 3.0])), [0.0, 0.0, 3.0]), "relu: check the negatives"
p = softmax(np.array([[1.0, 2.0, 3.0], [1000.0, 1000.0, 1000.0]]))
assert np.all(np.isfinite(p)), "softmax: overflow, did you subtract each row's maximum?"
assert np.allclose(p.sum(axis=1), 1.0), "softmax: every row must sum to 1"
assert np.allclose(p[0], [0.09003057, 0.24472847, 0.66524096]), "softmax: wrong values"
loss, d = cross_entropy(np.array([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]]), np.array([0, 1]))
assert np.isclose(loss, 0.3185, atol=1e-3), f"cross-entropy: expected ≈0.3185, got {loss:.4f}"
assert np.allclose(d.sum(axis=1), 0.0), "gradient: every row of (P - Y) must sum to 0"
assert np.isclose(d[0, 0], (softmax(np.array([[2.0, 1.0, 0.1]]))[0, 0] - 1) / 2), "gradient: did you divide by N?"
print("✅ 3.1 correct")

### 3.2 ✍️ The network
A class with three methods for you to complete:
- `forward` stores what backprop will need: each layer's input in `self.A` and the pre-activations in
  `self.Z`. The last layer does **not** apply ReLU.
- `backward` applies the equations from section 2, from the last layer to the first.
- `step` updates the parameters with gradient descent: $\theta \leftarrow \theta - \eta\,\nabla_\theta \mathcal{L}$.

Initialization is already done with **He** ($\sigma = \sqrt{2 / n_{\text{in}}}$), designed for ReLU: it keeps
the variance of the activations roughly constant across layers. If you initialize everything to zero, every
neuron in a layer receives the same gradient and they never become different (**symmetry**).

In [ ]:
class NeuralNetwork:
    """Fully connected MLP: ReLU in hidden layers, logits at the output."""

    def __init__(self, sizes, seed=0):
        g = np.random.default_rng(seed)
        self.W = [g.normal(0.0, np.sqrt(2.0 / n_in), (n_in, n_out))
                  for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros((1, n_out)) for n_out in sizes[1:]]

    def forward(self, X):
        self.A, self.Z = [X], []
        for layer, (W, b) in enumerate(zip(self.W, self.b)):
            # TODO: compute Z from the input self.A[-1], store it in self.Z and,
            #       unless this is the last layer, store relu(Z) in self.A
            raise NotImplementedError
        return Z

    def backward(self, dZ):
        """Take dL/dlogits and return the gradients (dW, db) of every layer."""
        dW, db = [None] * len(self.W), [None] * len(self.b)
        for layer in reversed(range(len(self.W))):
            # TODO 1: dW[layer] and db[layer] from dZ and self.A[layer]  (db with keepdims=True)
            # TODO 2: if layer > 0, propagate dZ to the previous layer (use self.W[layer] and self.Z[layer - 1])
            raise NotImplementedError
        return dW, db

    def step(self, dW, db, lr):
        # TODO: gradient descent on every W and b
        raise NotImplementedError

    def predict(self, X):
        return self.forward(X).argmax(axis=1)

In [ ]:
# ✅ Check 3.2 (shapes)
net = NeuralNetwork([2, 64, 64, 3])
logits = net.forward(X_train[:10])
assert logits.shape == (10, 3), f"forward: logits must be (10, 3), not {logits.shape}"
assert len(net.A) == 3 and len(net.Z) == 3, "forward: store 3 inputs in self.A and 3 pre-activations in self.Z"
dW, db = net.backward(np.ones((10, 3)))
assert [g.shape for g in dW] == [W.shape for W in net.W], "backward: every dW must have the shape of its W"
assert [g.shape for g in db] == [b.shape for b in net.b], "backward: every db must have the shape of its b"
print("Parameters:", sum(W.size + b.size for W, b in zip(net.W, net.b)))
print("✅ 3.2 shapes correct (section 4 checks the values)")

## 4. ✅ Numerical gradient check

Before training, we **verify** your `backward` by comparing it with centered finite differences:

$$
\frac{\partial \mathcal{L}}{\partial \theta} \approx \frac{\mathcal{L}(\theta + \varepsilon) - \mathcal{L}(\theta - \varepsilon)}{2\varepsilon}
$$

It uses a small network and `float64`. A relative error of $10^{-7}$ or smaller means your derivation is
correct; something like $10^{-2}$ points to a bug in backprop. This cell is already complete: read it
carefully, it is a tool you will use every time you code a gradient by hand.

In [ ]:
def gradient_check(net, X, y, eps=1e-5):
    loss, dlogits = cross_entropy(net.forward(X), y)
    dW, db = net.backward(dlogits)
    errors = []
    for params, grads in ((net.W, dW), (net.b, db)):
        for P, G in zip(params, grads):
            num = np.zeros_like(P)
            for i in np.ndindex(P.shape):
                original = P[i]
                P[i] = original + eps; plus, _ = cross_entropy(net.forward(X), y)
                P[i] = original - eps; minus, _ = cross_entropy(net.forward(X), y)
                P[i] = original
                num[i] = (plus - minus) / (2 * eps)
            errors.append(np.abs(num - G).max() / max(np.abs(num).max() + np.abs(G).max(), 1e-12))
    return max(errors)

error = gradient_check(NeuralNetwork([2, 5, 4, 3], seed=1), X_train[:20], y_train[:20])
print(f"Max relative error: {error:.2e}")
assert error < 1e-7, "backward does not match the numerical gradient: check the transposes and the ReLU derivative"
print("✅ 4 your backpropagation is correct")

## 5. ✍️ Training with mini-batches

The epoch loop and the shuffling are already written. You complete **one training step** on a
mini-batch: **forward → loss → backward → step**. These are exactly the steps PyTorch performs under the hood.

With `[2, 64, 64, 3]`, `lr=0.2` and 300 epochs you should exceed **95 % test accuracy**.

In [ ]:
def train_step(net, Xb, yb, lr):
    """One gradient-descent step on the mini-batch (Xb, yb). Returns the loss."""
    # TODO: forward → cross_entropy → backward → step. Return the loss.
    raise NotImplementedError

def accuracy(net, X, y):
    return (net.predict(X) == y).mean()

def train(net, X, y, epochs=300, lr=0.2, batch_size=32, seed=0):
    g = np.random.default_rng(seed)
    history = []
    for epoch in range(epochs):
        order = g.permutation(len(X))
        total = 0.0
        for start in range(0, len(X), batch_size):
            batch = order[start:start + batch_size]
            total += train_step(net, X[batch], y[batch], lr) * len(batch)
        history.append(total / len(X))
        if epoch % 50 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:3d} | loss {history[-1]:.4f} | train accuracy {accuracy(net, X, y):.3f}")
    return history

net = NeuralNetwork([2, 64, 64, 3], seed=0)
history_np = train(net, X_train, y_train)
acc_np = accuracy(net, X_test, y_test)
print(f"\nTest accuracy: {acc_np:.3f}")

plt.plot(history_np)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.grid(True)
plt.title("Learning curve (NumPy)"); plt.show()

In [ ]:
# ✅ Check 5
assert history_np[-1] < history_np[0] / 5, "the loss barely drops: check the sign of the update in step()"
assert acc_np > 0.95, f"test accuracy {acc_np:.3f}: it should exceed 0.95"
print("✅ 5 your network learns the spirals")

### 5.1 Decision boundaries: why do we need a hidden layer?

A network **without hidden layers** (`[2, 3]`) is multinomial logistic regression: it can only draw
**straight** boundaries. Stacking layers with ReLU lets the network compose many linear regions and
approximate boundaries as curved as the spirals. Run the cell and compare.

In [ ]:
def plot_boundary(predict, X, y, title, ax):
    xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 300), np.linspace(-1.2, 1.2, 300))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    ax.contourf(xx, yy, predict(grid).reshape(xx.shape), alpha=0.3, cmap="viridis")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="viridis", s=10)
    ax.set_title(title); ax.set_aspect("equal")

linear = NeuralNetwork([2, 3], seed=0)
_ = train(linear, X_train, y_train, epochs=100, lr=0.2)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_boundary(linear.predict, X_test, y_test,
              f"Linear [2, 3] · test {accuracy(linear, X_test, y_test):.2f}", axes[0])
plot_boundary(net.predict, X_test, y_test,
              f"MLP [2, 64, 64, 3] · test {accuracy(net, X_test, y_test):.2f}", axes[1])
plt.show()

## 6. The same network in PyTorch

Now you rewrite everything with PyTorch. What **disappears** is exactly what took the most work:
`backward` (handled by `autograd`), initialization (`nn.Linear`), the loss (`nn.CrossEntropyLoss`, which
already includes softmax) and the update (`torch.optim.SGD`).

### 6.1 ✍️ Build the model
Complete `torch_mlp` so that, given a list of sizes such as `[2, 64, 64, 3]`, it returns an `nn.Sequential`
with `nn.Linear` + `nn.ReLU` for each layer, **without** a ReLU after the last one.

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)

def torch_mlp(sizes):
    # TODO: build the list of layers and return nn.Sequential(*layers)
    raise NotImplementedError

model = torch_mlp([2, 64, 64, 3])
print(model)

# ✅ Check 6.1
assert [type(m).__name__ for m in model] == ["Linear", "ReLU", "Linear", "ReLU", "Linear"], \
    "the model must be Linear, ReLU, Linear, ReLU, Linear"
print("✅ 6.1 correct")

### 6.2 ✅ Do your gradients match PyTorch's?

This cell copies the weights of your NumPy network into the PyTorch model and compares `autograd`'s
gradients with **your** `backward`. Careful: `nn.Linear` stores weights as $(n_{\text{out}} \times n_{\text{in}})$
and computes $x W^\top + b$, which is why we **transpose**. It works in `float64` to compare precisely.

If everything is right, you will see differences of $10^{-16}$ or smaller: machine round-off.

In [ ]:
ref_net = NeuralNetwork([2, 64, 64, 3], seed=7)
twin = torch_mlp([2, 64, 64, 3]).double()
linears = [m for m in twin if isinstance(m, nn.Linear)]
with torch.no_grad():
    for layer, W, b in zip(linears, ref_net.W, ref_net.b):
        layer.weight.copy_(torch.from_numpy(W.T))
        layer.bias.copy_(torch.from_numpy(b.ravel()))

Xb, yb = X_train[:64], y_train[:64]

# NumPy: your backward
loss_np, dlogits = cross_entropy(ref_net.forward(Xb), yb)
dW, db = ref_net.backward(dlogits)

# PyTorch: autograd
loss_t = nn.functional.cross_entropy(twin(torch.from_numpy(Xb)), torch.from_numpy(yb))
loss_t.backward()

print(f"loss NumPy {loss_np:.8f} | PyTorch {loss_t.item():.8f}")
for i, layer in enumerate(linears):
    diff = max(np.abs(layer.weight.grad.numpy().T - dW[i]).max(),
               np.abs(layer.bias.grad.numpy() - db[i].ravel()).max())
    print(f"layer {i}: max gradient difference = {diff:.1e}")
    assert diff < 1e-10, f"layer {i}: your gradients do not match autograd"
print("✅ 6.2 your backward computes the same thing as autograd")

### 6.3 ✍️ Training in PyTorch
Same hyperparameters as in NumPy (SGD, `lr=0.2`, batches of 32, 300 epochs). Complete the training step
with the four steps you will repeat in every PyTorch project:
`zero_grad` → `forward` + `loss` → `backward` → `step`.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

Xtr_t = torch.tensor(X_train, dtype=torch.float32); ytr_t = torch.tensor(y_train)
Xte_t = torch.tensor(X_test, dtype=torch.float32);  yte_t = torch.tensor(y_test)
loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)

model = torch_mlp([2, 64, 64, 3])
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

def torch_step(xb, yb):
    """One PyTorch training step. Returns the loss as a tensor."""
    # TODO: the four PyTorch steps using model, loss_fn and optimizer
    raise NotImplementedError

history_t = []
for epoch in range(300):
    model.train()
    total = 0.0
    for xb, yb in loader:
        total += torch_step(xb, yb).item() * len(xb)
    history_t.append(total / len(loader.dataset))

model.eval()
with torch.no_grad():
    acc_t = (model(Xte_t).argmax(dim=1) == yte_t).float().mean().item()
print(f"Test accuracy — NumPy: {acc_np:.3f} | PyTorch: {acc_t:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(history_np, label="NumPy"); axes[0].plot(history_t, label="PyTorch")
axes[0].set_yscale("log"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy")
axes[0].legend(); axes[0].grid(True)

def predict_torch(M):
    with torch.no_grad():
        return model(torch.tensor(M, dtype=torch.float32)).argmax(dim=1).numpy()

plot_boundary(predict_torch, X_test, y_test, f"PyTorch · test {acc_t:.2f}", axes[1])
plt.show()

# ✅ Check 6.3
assert acc_t > 0.95, f"test accuracy {acc_t:.3f}: it should exceed 0.95"
print("✅ 6.3 your PyTorch network learns the spirals")

The curves will not be identical because initialization and batch order differ (PyTorch uses a uniform
Kaiming initialization, not the He normal), but both networks should reach a similar result.

## Summary
| Step | NumPy (from scratch) | PyTorch |
|---|---|---|
| Parameters | `W`, `b` lists with He initialization | `nn.Linear` inside `nn.Sequential` |
| Forward | `A @ W + b`, `relu` | `model(x)` |
| Loss | stable `softmax` + cross-entropy | `nn.CrossEntropyLoss` (takes logits) |
| Backward | hand-derived chain rule | `loss.backward()` (autograd) |
| Update | `W -= lr * dW` | `optimizer.step()` |
| Verification | numerical gradient check | — (we trust autograd) |

Writing backprop by hand once is what lets you **debug** real networks: when a loss does not go down,
you know what to look for (shapes, signs, gradient scales, initialization).

## ✏️ Extra exercises
1. **Tanh**: replace ReLU with $\tanh$ in your network (its derivative is $1 - \tanh^2$). Re-run the
   gradient check and compare convergence speed.
2. **L2 regularization**: add $\frac{\lambda}{2}\sum \lVert W \rVert^2$ to the loss and $\lambda W$ to each `dW`.
   Verify it with the gradient check and watch the boundary smooth out with $\lambda = 10^{-3}$ and $10^{-2}$.
3. **Momentum and Adam**: implement both optimizers in NumPy and compare their curves with SGD.
   Check that they match `torch.optim.SGD(momentum=0.9)` and `torch.optim.Adam`.
4. **Capacity**: what is the smallest network (width and depth) that exceeds 95 % test accuracy?
   What happens with 1,000 neurons and little data (`n_per_class=30`)?
5. **Initialization**: train with all weights set to zero and with $\sigma = 1$. Explain what you see.
6. **Real data**: apply your `NeuralNetwork` to `sklearn.datasets.load_digits` (8×8 images, 10 classes).
   Remember to standardize the inputs.